In [1]:
import pandas as pd

# 1. 정제된 재료 데이터 로드
# 웰니스 데이터는 중복을 제거한 최종 19,666건 파일을 사용합니다.
wellness = pd.read_csv('cleaned_wellness.csv')
chatbot = pd.read_csv('cleaned_chatbot.csv')
subject = pd.read_csv('cleaned_subject.csv')

# 2. 일상 데이터 통합 풀(Pool) 생성 (DS1, DS4용)
daily_pool = pd.concat([chatbot, subject], ignore_index=True)

print("--- 데이터 로드 및 준비 완료 ---")
print(f"우울 데이터(Wellness): {len(wellness)}건")
print(f"일상 데이터(Chatbot): {len(chatbot)}건")
print(f"일상 데이터(Subject): {len(subject)}건")
print(f"통합 일상 데이터 풀: {len(daily_pool)}건\n")

# 3. 데이터셋 구축 및 파일 저장 함수
def build_and_save_dataset(wellness_df, daily_source_df, sample_size, file_name):
    """
    wellness_df: 우울 데이터프레임
    daily_source_df: 샘플링할 일상 데이터 원천
    sample_size: 일상 데이터에서 뽑을 개수
    file_name: 저장할 파일 이름
    """
    # 일상 데이터 무작위 샘플링 (재현성을 위해 random_state=42 고정)
    daily_sampled = daily_source_df.sample(n=sample_size, random_state=42)
    
    # 우울 데이터와 샘플링된 일상 데이터 결합
    combined_df = pd.concat([wellness_df, daily_sampled], ignore_index=True)
    
    # 모델 학습을 위해 전체 데이터를 무작위로 섞음 (Shuffle)
    final_dataset = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # CSV 파일로 저장 (한글 깨짐 방지 utf-8-sig)
    final_dataset.to_csv(file_name, index=False, encoding='utf-8-sig')
    
    print(f"✅ {file_name} 생성 완료")
    print(f"   (총 {len(final_dataset)}건 = 우울 {len(wellness_df)} + 일상 {sample_size})")
    return final_dataset

# 4. 논문 실험 설계에 따른 4가지 데이터셋 생성 실행

# Dataset 1: 우울(19,666) + 일상(23,000 / 전체 풀에서 추출)
ds1 = build_and_save_dataset(wellness, daily_pool, 23000, 'dataset_1.csv')

# Dataset 2: 우울(19,666) + 일상(18,000 / 주제별 데이터에서 추출)
ds2 = build_and_save_dataset(wellness, subject, 18000, 'dataset_2.csv')

# Dataset 3: 우울(19,666) + 일상(1,000 / 주제별 데이터에서 추출)
ds3 = build_and_save_dataset(wellness, subject, 1000, 'dataset_3.csv')

# Dataset 4: 우울(19,666) + 일상(6,290 / 전체 풀에서 추출)
ds4 = build_and_save_dataset(wellness, daily_pool, 6290, 'dataset_4.csv')

print("\n✨ 모든 실험용 데이터셋 파일이 성공적으로 저장되었습니다.")

--- 데이터 로드 및 준비 완료 ---
우울 데이터(Wellness): 19666건
일상 데이터(Chatbot): 5261건
일상 데이터(Subject): 1370984건
통합 일상 데이터 풀: 1376245건

✅ dataset_1.csv 생성 완료
   (총 42666건 = 우울 19666 + 일상 23000)
✅ dataset_2.csv 생성 완료
   (총 37666건 = 우울 19666 + 일상 18000)
✅ dataset_3.csv 생성 완료
   (총 20666건 = 우울 19666 + 일상 1000)
✅ dataset_4.csv 생성 완료
   (총 25956건 = 우울 19666 + 일상 6290)

✨ 모든 실험용 데이터셋 파일이 성공적으로 저장되었습니다.
